<a href="https://colab.research.google.com/github/dashang/RAG/blob/main/18Oct2025_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# You may need to install these libraries first:
# pip install sentence-transformers scikit-learn numpy

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# --- 1. Sample Data & Query ---
chunks = [
    "Applicants must be at least 21 years old.", # Relevant
    "Minimum monthly salary is ₹25,000.", # Relevant
    "Applicants must have a credit score above 700.", # Relevant
    "The age requirement is that applicants must be at least 21.", # Redundant
    "A credit score of 700 or higher is mandatory.", # Redundant
    "Personal loans are not available to self-employed applicants.", # Slightly less relevant
    "EMI defaults attract a penalty of 2% per month.", # Irrelevant
]
query = "What are the age, salary, and credit score requirements for a personal loan?"

# --- 2. Embedding ---
model = SentenceTransformer('all-MiniLM-L6-v2')
chunk_embeddings = model.encode(chunks)
query_embedding = model.encode([query])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# --- 3. Vanilla Top-k Retrieval ---
def vanilla_top_k(query_embedding, chunk_embeddings, chunks, k=3):
    similarities = cosine_similarity(query_embedding, chunk_embeddings)[0]
    top_k_idx = similarities.argsort()[-k:][::-1]
    return [chunks[i] for i in top_k_idx], similarities[top_k_idx]

vanilla_results, vanilla_scores = vanilla_top_k(query_embedding, chunk_embeddings, chunks, k=3)
print("--- Vanilla Top-3 Results (Often Redundant) ---")
for text, score in zip(vanilla_results, vanilla_scores):
    print(f"  - (Score: {score:.2f}) {text}")


# --- 4. MMR-based Retrieval ---
def mmr(query_embedding, chunk_embeddings, chunks, k=3, lambda_param=0.5):
    """Returns k chunks using Maximum Marginal Relevance."""
    query_chunk_similarities = cosine_similarity(query_embedding, chunk_embeddings)[0]

    selected_indices = []
    candidates_indices = list(range(len(chunks)))

    best_idx = np.argmax(query_chunk_similarities)
    selected_indices.append(best_idx)
    candidates_indices.remove(best_idx)

    for _ in range(k - 1):
        if not candidates_indices: break

        mmr_scores = []
        candidate_embeddings = chunk_embeddings[candidates_indices]

        for i, cand_idx in enumerate(candidates_indices):
            relevance_to_query = query_chunk_similarities[cand_idx]

            selected_embeddings = chunk_embeddings[selected_indices]
            max_similarity_to_selected = np.max(cosine_similarity(candidate_embeddings[i:i+1], selected_embeddings))

            score = lambda_param * relevance_to_query - (1 - lambda_param) * max_similarity_to_selected
            mmr_scores.append(score)

        best_candidate_idx = candidates_indices[np.argmax(mmr_scores)]
        selected_indices.append(best_candidate_idx)
        candidates_indices.remove(best_candidate_idx)

    return [chunks[i] for i in selected_indices]

mmr_results = mmr(query_embedding, chunk_embeddings, chunks, k=3, lambda_param=0.7)
print("\n--- MMR Top-3 Results (Relevant and Diverse) ---")
for text in mmr_results:
    print(f"  - {text}")


--- Vanilla Top-3 Results (Often Redundant) ---
  - (Score: 0.57) Applicants must have a credit score above 700.
  - (Score: 0.57) A credit score of 700 or higher is mandatory.
  - (Score: 0.49) Personal loans are not available to self-employed applicants.

--- MMR Top-3 Results (Relevant and Diverse) ---
  - Applicants must have a credit score above 700.
  - Personal loans are not available to self-employed applicants.
  - Minimum monthly salary is ₹25,000.
